<a href="https://colab.research.google.com/github/Ayu-sshhhhh/Food-Vision-Classifier/blob/main/Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Thu Apr  9 13:52:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Importing dataset

10% of 10-food-classes from Food 101 - https://www.kaggle.com/dansbecker/food-101

https://github.com/mrdbourke/tensorflow-deep-learning/blob/main/04_transfer_learning_in_tensorflow_part_1_feature_extraction.ipynb

In [1]:
import zipfile
!wget https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip
zip_ref = zipfile.ZipFile("10_food_classes_10_percent.zip")
zip_ref.extractall()
zip_ref.close()

--2026-04-09 13:56:54--  https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.141.207, 142.251.2.207, 74.125.137.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.250.141.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 168546183 (161M) [application/zip]
Saving to: ‘10_food_classes_10_percent.zip’

10_food_classes_10_ 100%[===================>] 160.74M   158MB/s    in 1.0s    

2026-04-09 13:56:56 (158 MB/s) - ‘10_food_classes_10_percent.zip’ saved [168546183/168546183]



# Glimpse of dataset

In [2]:
import os
for dirpath, dirnames, filenames in os.walk("10_food_classes_10_percent"):
  print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath} ")

There are 2 directories and 0 images in '10_food_classes_10_percent 
There are 10 directories and 0 images in '10_food_classes_10_percent/test 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/ramen 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/ice_cream 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/grilled_salmon 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/hamburger 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/sushi 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/chicken_curry 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/steak 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/pizza 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/chicken_wings 
There are 0 directories and 250 images in '10_food_classes_10_percent/test/fried_rice

In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMAGE_SHAPE = (224,224)
BATCH_SIZE = 32
EPOCHS = 5

train_dir = "10_food_classes_10_percent/train/"
test_dir = "10_food_classes_10_percent/test/"

train_datagen = ImageDataGenerator(rescale=1/255.)
test_datagen = ImageDataGenerator(rescale=1/255.)
print("Training Images : ")
train_data_10_percent = train_datagen.flow_from_directory(train_dir,
                                                          target_size = IMAGE_SHAPE,
                                                          batch_size = BATCH_SIZE,
                                                          class_mode = 'categorical')
print("Testing Images : ")
test_data_10_percent = test_datagen.flow_from_directory(test_dir,
                                                        target_size = IMAGE_SHAPE,
                                                        batch_size = BATCH_SIZE,
                                                        class_mode = 'categorical')

Training Images : 
Found 750 images belonging to 10 classes.
Testing Images : 
Found 2500 images belonging to 10 classes.


# Setting up callbacks

In [6]:
# creating Tensorboard callbacks ( and functionizing it)
import datetime
def create_tensorboard_callback(dir_name, experiment_name):
  log_dir = dir_name +"/" + experiment_name +"/"+ datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
  tensorboard_callback = tf.keras.callbacks.Tensorboard(log_dir=log_dir)
  print(f"Saving TensorBoard log files to : {log_dir}")
  return tensorboard_callback